
# FEVER → DeBERTa-v3 Fact Verification Judge


## 1. Imports and reproducibility

In [ ]:
import pyarrow
print("pyarrow:", pyarrow.__version__)

import datasets
print("datasets:", datasets.__version__)

from datasets import load_dataset
print("datasets import OK")

In [ ]:

import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 2. Load FEVER-NLI pairs

In [ ]:

DATASET_NAME = "minko186/fever-nli-3class-split"
raw = load_dataset(DATASET_NAME)
raw


In [ ]:

for split in raw.keys():
    print("\nSplit:", split)
    print("Columns:", raw[split].column_names)
    print("Rows:", len(raw[split]))
    print(raw[split][0])


### Inspect labels before training

In [ ]:

for split in raw.keys():
    df_tmp = raw[split].to_pandas()
    print(f"\n{split} numeric labels:")
    print(df_tmp["label"].value_counts(dropna=False).sort_index())
    if "label_str" in df_tmp.columns:
        print(f"\n{split} label_str:")
        print(df_tmp["label_str"].value_counts(dropna=False))


In [ ]:

sample_df = raw["train"].to_pandas()
for lab in sorted(sample_df["label"].unique()):
    print("\n" + "="*90)
    print("NUMERIC LABEL:", lab)
    cols = ["text1", "text2", "label"]
    if "label_str" in sample_df.columns:
        cols.append("label_str")
    display(sample_df[sample_df["label"] == lab][cols].head(3))



## 3. Map labels to our Judge task


In [ ]:

SOURCE_TO_TARGET = {
    0: 0,  
    2: 1,   
}

id2label = {
    0: "UNSUPPORTED",
    1: "SUPPORTED",
}

label2id = {
    "UNSUPPORTED": 0,
    "SUPPORTED": 1,
}

def normalize_example(example):
    return {
        "claim": example["text1"],
        "evidence": example["text2"],
        "labels": SOURCE_TO_TARGET[int(example["label"])],
    }

dataset = raw.map(normalize_example)
keep = {"claim", "evidence", "labels"}
for split in dataset.keys():
    remove_cols = [x for x in dataset[split].column_names if x not in keep]
    dataset[split] = dataset[split].remove_columns(remove_cols)

dataset



## 4. Start with a smaller run


In [ ]:
FAST_RUN = True

if FAST_RUN:

    train_df = dataset["train"].to_pandas()

    unsupported_df = train_df[
        train_df["labels"] == 0
    ]

    supported_df = train_df[
        train_df["labels"] == 1
    ]

    train_n = 10000
    n_per_class = train_n // 2

    unsupported_sample = unsupported_df.sample(
        n=min(n_per_class, len(unsupported_df)),
        random_state=SEED
    )

    supported_sample = supported_df.sample(
        n=min(n_per_class, len(supported_df)),
        random_state=SEED
    )

    balanced_train_df = pd.concat([
        unsupported_sample,
        supported_sample
    ])

    balanced_train_df = balanced_train_df.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    train_small = Dataset.from_pandas(
        balanced_train_df,
        preserve_index=False
    )

    eval_name = (
        "validation"
        if "validation" in dataset
        else "test"
    )

    eval_n = min(
        2000,
        len(dataset[eval_name])
    )

    validation_small = (
        dataset[eval_name]
        .shuffle(seed=SEED)
        .select(range(eval_n))
    )

    work_ds = DatasetDict({
        "train": train_small,
        "validation": validation_small
    })


else:

    eval_name = (
        "validation"
        if "validation" in dataset
        else "test"
    )

    work_ds = DatasetDict({
        "train": dataset["train"],
        "validation": dataset[eval_name]
    })


print(work_ds)


In [ ]:
from collections import Counter

print("TRAIN:")
print(Counter(work_ds["train"]["labels"]))

print("\nVALIDATION:")
print(Counter(work_ds["validation"]["labels"]))

In [ ]:

for split in work_ds.keys():
    s = pd.Series(work_ds[split]["labels"])
    print("\n", split)
    print(s.value_counts().sort_index().rename(index=id2label))



## 5. Tokenization

In [ ]:

#MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["evidence"],
        batch["claim"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = work_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["claim", "evidence"],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized


## 6. Build DeBERTa-v3 3-class classifier

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    attn_implementation="eager",
)

print("Total parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")

print("Trainable parameters:",
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 7. Metrics: Accuracy + Macro-F1 + per-class F1

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    per_class = f1_score(
        labels, preds,
        labels=[0,1],
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "f1_unsupported": per_class[0],
        "f1_supported": per_class[1],
    }



## 8. Training configuration


In [ ]:

OUTPUT_DIR = "./deberta_fever_judge"

training_args = TrainingArguments(
     output_dir="./deberta_debug",

    learning_rate=1e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,

    num_train_epochs=5,
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=1,
    logging_first_step=True,
    logging_nan_inf_filter=False,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,  
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 9. Train

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

print("TRAIN LABELS:")
print(Counter(tokenized["train"]["labels"]))

print("\nVALIDATION LABELS:")
print(Counter(tokenized["validation"]["labels"]))

pred = trainer.predict(tokenized["validation"])

y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)

print("\nTRUE:")
print(Counter(y_true))

print("\nPREDICTED:")
print(Counter(y_pred))

print("\nAny NaN logits?")
print(np.isnan(pred.predictions).any())

print("\nAny Inf logits?")
print(np.isinf(pred.predictions).any())

## 10. Evaluate

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:

pred_output = trainer.predict(tokenized["validation"])

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["UNSUPPORTED", "SUPPORTED"],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

pd.DataFrame(
    cm,
    index=["true_U", "true_S"],
    columns=["pred_U", "pred_S"],
)

## 11. Save the trained Judge

In [ ]:

SAVE_DIR = "./fever_deberta_judge"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)



## 12. Single claim + evidence inference

This is the function you can plug into your hallucination-detection pipeline.


In [ ]:

import torch.nn.functional as F

def judge(claim, evidence, model=model, tokenizer=tokenizer):
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        evidence,
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]

    pred_id = int(torch.argmax(probs).item())

    return {
        "label": id2label[pred_id],
        "confidence": float(probs[pred_id].item()),
        "probabilities": {
            id2label[i]: float(probs[i].item())
            for i in range(2)
        },
    }


In [ ]:

judge(
    claim="The Eiffel Tower is located in Paris.",
    evidence="The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.",
)


In [ ]:

judge(
    claim="The Eiffel Tower is located in London.",
    evidence="The Eiffel Tower is located in Paris, France.",
)


In [ ]:

judge(
    claim="Albert Einstein enjoyed eating sushi.",
    evidence="Albert Einstein developed the theory of relativity and received the 1921 Nobel Prize in Physics.",
)


## 13. Batch inference for your retrieved claims

In [ ]:

def judge_dataframe(df, claim_col="claim", evidence_col="evidence"):
    rows = []

    for _, row in df.iterrows():

        result = judge(
            str(row[claim_col]),
            str(row[evidence_col])
        )

        rows.append({
            **row.to_dict(),
            "judge_label": result["label"],
            "judge_confidence": result["confidence"],
            "p_unsupported": result["probabilities"]["UNSUPPORTED"],
            "p_supported": result["probabilities"]["SUPPORTED"],
        })

    return pd.DataFrame(rows)
